In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# [Baseline1 코드와의 차이점 1]
learning rate scheduler를 사용하였습니다.
>scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

***

https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate

Learning rate scheduler는 학습이 진행됨에 따라 epoch 또는 iteration 간에 학습률을 조정하는 사전 정의된 프레임워크입니다. Learning rate scheduler에 대한 가장 일반적인 두 가지 기술은 다음과 같습니다.

- 일정한 Learning rate: 이름에서 알 수 있듯이 학습률을 초기화하고 훈련 중에 변경하지 않습니다.
- Learning rate 감소: 초기 학습률을 선택한 다음 스케줄러에 따라 점차적으로 감소시킵니다.

훈련 초기에 학습률은 충분히 좋은 가중치에 도달하기 위해 크게 설정됩니다. 시간이 지남에 따라 이러한 가중치는 작은 학습률을 활용하여 더 높은 정확도에 도달하도록 미세 조정됩니다.

***
여러 Learning rate scheduler 중에서 지수적으로 learning rate가 감소하는 `ExponentialLR`을 사용하였습니다. 파라미터인 `gamma`는 감소율을 의미합니다.

![](https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fk.kakaocdn.net%2Fdn%2F4N7zv%2FbtqOSxJngX5%2FpTYdc3wYe94BzZvKzFtSbK%2Fimg.png)

In [2]:
# <!-- # [Baseline1 코드와의 차이점 2]
# 기존 resnet18 모델에 nn.Linear 모델을 추가하여 새로운 모델 resnetplus를 정의하였습니다.

# >def resnetplus(model): (아래코드참고) -->

In [3]:
import numpy as np # linear algebra
import pandas as pd
import random
import torch

seed = 1
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [4]:
import os
from PIL import Image
from tqdm import tqdm
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [6]:
train_path = "/kaggle/input/2022-ai-tp-18011886/train"
test_path = "/kaggle/input/2022-ai-tp-18011886/test"
all_label = os.listdir(train_path)

In [7]:
class train_dataset():
    def __init__(self):
        self.data_path = []
        self.label_idx = []
        
        for idx, label in enumerate(all_label):
            for data_path in tqdm(os.listdir(os.path.join(train_path,label))):
                path = os.path.join(label,data_path)
                self.data_path.append(path)
                self.label_idx.append(idx)
                
    def __getitem__(self,idx):
        transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((224,224)),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
        
        path = self.data_path[idx]
        img = Image.open(os.path.join(train_path,path))
        img = img.convert('RGB')
        img = transform(img)
        return img, self.label_idx[idx]
    
    def __len__(self):
        return len(self.data_path)

class test_dataset():
    def __init__(self):
        self.data_path = []
        
        for data_path in tqdm(os.listdir(os.path.join(test_path))):
            path = data_path
            self.data_path.append(path)
            
        self.data_path.sort()
                
    def __getitem__(self,idx):
        transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((224,224)),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
        
        path = self.data_path[idx]
        img = Image.open(os.path.join(test_path,path))
        img = img.convert('RGB')
        img = transform(img)
        return img, 0
    
    def __len__(self):
        return len(self.data_path)

In [8]:
train_dataset = train_dataset()
test_dataset = test_dataset()
train_loader = DataLoader(train_dataset, batch_size=100,
                         shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=100,
                         shuffle=False)

100%|██████████| 1312/1312 [00:00<00:00, 816700.33it/s]


In [9]:
# from torchvision.models import efficientnet_b0
# import torch.nn as nn

# model = efficientnet_b0(pretrained=True)
# model.classifier[1] = nn.Linear(in_features=1280, out_features=4, bias=True) 
# model = model.to(device)

In [10]:
import torchvision.models as models

model = models.resnet18(pretrained=True)
model.fc = torch.nn.Linear(512, 4, bias=True)
torch.nn.init.xavier_normal_(model.fc.weight)
model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


  0%|          | 0.00/44.7M [00:00<?, ?B/s]

In [11]:
# import torchvision.models as models

# model = models.resnet18(pretrained=True)

# model.layer1.requires_grad = False
# model.layer2.requires_grad = False
# model.layer3.requires_grad = False
# model.layer4.requires_grad = False

# def resnetplus(model):
#     linear1 = torch.nn.Linear(1000,256)
#     linear2 = torch.nn.Linear(256,4)
    
#     torch.nn.init.xavier_normal_(linear1.weight)
#     torch.nn.init.xavier_normal_(linear2.weight)

#     relu = torch.nn.LeakyReLU()
#     dropout = torch.nn.Dropout(0.2)
#     res_model = torch.nn.Sequential(model,
#                                    linear1,relu,dropout,
#                                    linear2)
    
#     return res_model

# model = resnetplus(model).to(device)

In [12]:
# import torchvision.models as models

# model = models.vgg16(pretrained=True)
# model.classifier[6] = torch.nn.Linear(4096, 4, bias=True)
# torch.nn.init.xavier_normal_(model.classifier[6].weight)
# model.to(device)

In [13]:
loss = torch.nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

In [14]:
model.train()

epochs = 10

for epoch in tqdm(range(epochs)):
    avg_cost = 0
    
    for x,y in train_loader:
        #import pdb;pdb.set_trace()
        x = x.to(device)
        y = y.to(device)
        
        optimizer.zero_grad()
        out = model(x).to(device)
        cost = loss(out, y).to(device)
        cost.backward()
        optimizer.step()
        
        scheduler.step()
        avg_cost +=cost.item()/len(train_loader)
        
    print(epoch+1,avg_cost)

 10%|█         | 1/10 [04:09<37:27, 249.67s/it]

1 1.143463365927986


 20%|██        | 2/10 [07:10<27:54, 209.30s/it]

2 0.9255411624908452


 30%|███       | 3/10 [10:11<22:54, 196.32s/it]

3 0.9247068710949109


... (10 outputs left out)

 90%|█████████ | 9/10 [27:57<02:58, 178.93s/it]

9 0.924418740168862


100%|██████████| 10/10 [30:54<00:00, 185.44s/it]

10 0.9231057073758998


In [15]:
with torch.no_grad():
    model.eval()
    corr_pred = list()
    test = list()
    for x,_ in test_loader:
        x = x.to(device)
        out = model(x).to(device)
        pred = torch.argmax(out,dim=1).cpu().detach().numpy()
        corr_pred.extend(pred)        
    
    for i in range(len(corr_pred)):
        test.append(all_label[corr_pred[i]])

In [16]:
submit = pd.read_csv('/kaggle/input/2022-ai-tp-18011886/sample_submit.csv',encoding='cp949')
submit['label'] = test
submit.to_csv('submit.csv',index=False,encoding='cp949')